# microWakeWord training (Kaggle, private Python 3.11)

Trains an **on-device** wake word and produces `<name>.tflite` + `<name>.json`, the pair ESPHome's
`micro_wake_word` (Voice PE) and Android apps load. Also `<name>_cutoffs.txt`: false wakes per hour and misses at each cutoff.

**Before running**, right sidebar → **Settings**: **Accelerator** = GPU T4 x2, **Internet** = On.

1. **Step 1**: edit `CONFIG` (wake word, variants, language), run, and listen to the test clips.
2. **Step 2**: downloads augmentation and negative datasets (several GB). Check the 📦 Data check.
3. **Step 3**: generates samples, builds features, trains, and saves the files to `/kaggle/working`.

**Your own recordings (optional, the best help in a noisy room).** Make a Kaggle dataset with two folders of
16 kHz mono WAVs: `positive/` (you saying the wake word, on the device that will listen) and `negative/` (things that
wrongly woke it: the assistant's own voice, the TV). Attach it (right sidebar → **Add Input**): `MY_CLIPS: 'auto'` finds it,
or put its path there. Step 1 says how many clips it found. Clips a device saved when it woke can go in as they are: the notebook finds the word in each
positive clip and keeps just that part (`MY_CLIPS_FIND_WORD`). Listen first: a wrong file in `positive/` teaches the wrong thing.

For unattended runs use **Save Version → Save & Run All (Commit)** and download from the version's **Output** tab.

## Step 1: Settings, setup, and test pronunciations

In [10]:
# STEP 1: Settings, Python 3.11 setup, and test pronunciations
# Edit CONFIG, then run. The first run takes several minutes (Python 3.11 + packages). Re-runs are quick.

CONFIG = {
    'WAKE_WORD_NAME': 'esmeralda',         # file/model name
    'WAKE_WORD_DISPLAY': 'Esmeralda',     # name shown in the wake word dropdown
    'CLIP_DURATION_MS': 1500,              # 1500 for a short phrase, 1800 for longer. NOT 2000 (stride bug)
    # Goes into the JSON. 'auto' picks the lowest cutoff with at most TARGET_FALSE_WAKES_PER_HOUR in the
    # dinner party test (needs USE_DINNER_PARTY_EVAL) that still misses at most MAX_MISSES of the wake words; if
    # none does both, the one with the fewest false wakes within MAX_MISSES, with a warning. A number is used as
    # is. Official models use 0.97.
    'PROBABILITY_CUTOFF': 'auto',
    'TARGET_FALSE_WAKES_PER_HOUR': 0.5,
    'MAX_MISSES': 0.10,                    # a cutoff that misses more than this share of wake words is never picked
    'FALLBACK_CUTOFF': 0.85,               # used if 'auto' finds no table

    # Step 1 plays one clip of each TEST_VARIANT. Only VARIANTS are used for training.
    'TEST_VARIANTS': ['esmeralda', 'ezmeralda', 'ezmaralda'],
    'VARIANTS': ['esmeralda', 'ezmeralda', 'ezmaralda'],
    'SAMPLES_PER_COMBO': 200,              # per variant per voice (total = variants x voices x this)
    # Extra samples per variant from Piper's LibriTTS-R generator: ~900 speakers blended, far more voices than the
    # single-speaker models. 0 to skip. English only.
    'GENERATOR_SAMPLES': 3000,

    'LANGUAGE': 'en_US',                   # Piper voices: https://rhasspy.github.io/piper-samples/
    'VOICE_QUALITY': 'medium',

    # Hard negatives: words that sound close, said by every voice, so it learns they are NOT the wake word.
    'SOUNDALIKES': ['esme', 'emerald', 'emeralds', 'the emerald city', 'ramirez', 'rosalind', 'mirella',
                    'esperanza', 'esther', 'ezra', 'umbrella', 'is my ralph there'],
    'SOUNDALIKE_SAMPLES_PER_VOICE': 100,
    # The assistant's own voice (the Piper voice its replies are made in) saying everyday replies: what it hears of
    # itself. '' to skip. Avoid the wake word in HER_PHRASES.
    'HER_VOICE': 'en_GB-cori-medium',
    'HER_PHRASES': ['Turned on the lights', 'Turned off the kitchen lights', 'Good morning', 'Sorry, I didn\'t catch that',
                    'The weather today is sunny with a high of seventy two', 'Your timer is set for five minutes',
                    'Playing music in the living room', 'It is ten thirty', 'Done', 'I am not sure about that',
                    'The front door is locked', 'Here is the news', 'Okay, setting the temperature to sixty eight'],
    'HER_SAMPLES': 600,

    # Your own recordings (see the top of the notebook): 'auto' finds an attached dataset with positive/ or
    # negative/ folders of WAVs; or its path, e.g. '/kaggle/input/esmeralda-clips'; '' to skip.
    'MY_CLIPS': 'auto',
    'MY_CLIPS_REPEAT': 20,                 # each real clip is augmented this many times (there are few of them)
    # Training keeps the END of each positive clip (CLIP_DURATION_MS of it). A clip a device saved when it woke may
    # end well after the word (it woke late), so the word is found first: the loudest WORD_SECONDS of the clip, and
    # the clip cut to end just after it. False keeps your positives as they are.
    'MY_CLIPS_FIND_WORD': True,
    'MY_CLIPS_WORD_SECONDS': 0.8,          # about how long the wake word takes to say

    'NEGATIVE_CLASS_WEIGHT': 20,           # higher = fewer false triggers, but less sensitive
    'TRAINING_STEPS': 20000,               # at learning rate 0.001
    'FINE_TUNE_STEPS': 10000,              # then at 0.0001; 0 to skip
    'SPEC_AUGMENT': True,                  # hides random slices of time and pitch while training: sturdier in noise
    'EVAL_STEP_INTERVAL': 2000,
    'USE_DINNER_PARTY_EVAL': True,         # measures false wakes per hour; ~4 GB more RAM (Kaggle has plenty)
}

import os, sys, re, glob, json, shutil, subprocess
WORK = "/kaggle/temp/mww" if os.path.isdir("/kaggle/temp") else "/tmp/mww"
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
VPY = f"{WORK}/py311/bin/python"          # the private Python 3.11
UV = [sys.executable, "-m", "uv"]

def run(cmd, check=True, cwd=None, noise=()):
    # Run a command, stream its output live, optionally hide known-harmless log lines
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, "PYTHONUNBUFFERED": "1", "TF_CPP_MIN_LOG_LEVEL": "2",
                              "MPLBACKEND": "Agg"})  # Kaggle's inline plot setting breaks matplotlib in Python 3.11
    for line in p.stdout:
        if not any(n in line for n in noise):
            print(line, end="")
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f"❌ Command failed (exit {p.returncode}): {' '.join(map(str, cmd))[:150]}")
    return p.returncode

def count_wavs(d):
    return len(glob.glob(f"{d}/*.wav")) if os.path.isdir(d) else 0

NOTEBOOK = "2026-10-23c"                  # which notebook this is: Step 3 checks its saved settings came from it
CONFIG["NOTEBOOK"] = NOTEBOOK
print(f"📓 Notebook {NOTEBOOK}")

# Your own recordings: found and counted now, so a wrong path or a missing dataset shows in the first minute
def my_wavs(root, kind):
    return glob.glob(f"{root}/**/{kind}/*.wav", recursive=True) if root else []
if CONFIG["MY_CLIPS"] == "auto":
    roots = sorted({os.path.dirname(os.path.dirname(f)) for kind in ("positive", "negative")
                    for f in my_wavs("/kaggle/input", kind)})
    CONFIG["MY_CLIPS"] = roots[0] if roots else ""
    if len(roots) > 1:
        print(f"⚠️ More than one attached dataset has positive/ or negative/ folders; using {roots[0]}. Others: {roots[1:]}")
if CONFIG["MY_CLIPS"]:
    pos, neg = len(my_wavs(CONFIG["MY_CLIPS"], "positive")), len(my_wavs(CONFIG["MY_CLIPS"], "negative"))
    if pos + neg == 0:
        raise RuntimeError(f"❌ MY_CLIPS is {CONFIG['MY_CLIPS']!r} but it has no positive/ or negative/ WAVs. "
                           f"Attached: {os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else 'nothing'}")
    print(f"🎙️ Your recordings, {CONFIG['MY_CLIPS']}: {pos} positive, {neg} negative")
else:
    print("🎙️ No recordings of your own (none attached, or MY_CLIPS is '')")

with open(f"{WORK}/config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)
print(f"Wake word: {CONFIG['WAKE_WORD_DISPLAY']} ({CONFIG['WAKE_WORD_NAME']}), language {CONFIG['LANGUAGE']}")

import urllib.request
try:
    urllib.request.urlopen("https://huggingface.co", timeout=15)
except Exception:
    raise RuntimeError("❌ No internet. Right sidebar: Settings → Internet → On, then rerun.")
run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=False)
print(f"💾 Free disk in {WORK}: {shutil.disk_usage(WORK).free/1e9:.0f} GB")

# Packages for the private Python 3.11 (versions checked to resolve together; TF and torch share CUDA 12)
PKGS = ["tensorflow[and-cuda]==2.20.0", "torch==2.8.0", "torchaudio==2.8.0",
        "datasets==2.21.0", "piper-tts==1.3.0", "audiomentations==0.43.1", "setuptools<81",
        "soundfile", "pydub", "librosa", "scipy", "tqdm", "pyyaml",
        "audio-metadata @ git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f"]

MARKER = f"{WORK}/.py311_ready"
if not os.path.exists(MARKER):
    print("⚙️ One-time setup: installing uv and Python 3.11...")
    if run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=False) != 0:
        run([sys.executable, "-m", "pip", "install", "-q", "--break-system-packages", "uv"])
    if not os.path.exists(VPY):
        run(UV + ["venv", "--python", "3.11", f"{WORK}/py311"])
    if not os.path.exists("microWakeWord"):
        run(["git", "clone", "-q", "https://github.com/kahrendt/microWakeWord"])
    if not os.path.exists("piper-sample-generator"):
        run(["git", "clone", "-q", "https://github.com/rhasspy/piper-sample-generator"])
    print("📦 Installing packages into Python 3.11 (several minutes)...")
    run(UV + ["pip", "install", "--python", VPY] + PKGS + ["-e", "./microWakeWord"])
    open(MARKER, "w").write("ok")
    print("✅ Python 3.11 environment ready")
else:
    print("✅ Python 3.11 environment already set up")

# Repair: the old "webrtcvad" package overwrites microWakeWord's "webrtcvad-wheels" and needs pkg_resources
run(UV + ["pip", "uninstall", "-q", "--python", VPY, "webrtcvad"], check=False)
run(UV + ["pip", "install", "-q", "--python", VPY, "--reinstall-package", "webrtcvad-wheels",
          "webrtcvad-wheels", "setuptools<81", "audiomentations==0.43.1"], check=False)  # 0.33 lacks AddColorNoise
run([VPY, "-c", "import webrtcvad; webrtcvad.Vad(2); print('✅ webrtcvad OK')"], check=False)

if not shutil.which("ffmpeg"):          # needed to decode the FMA music files
    run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=False)

# Patch microWakeWord's stride assertion (from the original notebook)
utils_py = "microWakeWord/microwakeword/utils.py"
code = open(utils_py).read()
old = "            assert spectrogram.shape[0] % stride == 0"
if old in code:
    code = code.replace(old, "            remainder = spectrogram.shape[0] % stride\n"
                             "            if remainder != 0:\n"
                             "                spectrogram = spectrogram[:spectrogram.shape[0] - remainder]")
    open(utils_py, "w").write(code)
    print("✅ Stride patch applied")
else:
    print("✅ Stride patch already applied")

run([VPY, "-c", "import sys, tensorflow as tf; g = tf.config.list_physical_devices('GPU'); "
                "print('🐍 Python', sys.version.split()[0], '| TensorFlow', tf.__version__, '| GPU:', "
                "g[0].name if g else 'NONE: set Accelerator to GPU T4 x2 (training would be very slow)')"],
    noise=("cuda_dnn", "cuda_blas", "cuda_fft", "computation_placer", "absl::InitializeLog", "WARNING: All log"))

# --- Piper voices for the chosen language/quality ---
if not os.path.exists("voices.json"):
    urllib.request.urlretrieve("https://huggingface.co/rhasspy/piper-voices/resolve/main/voices.json", "voices.json")
all_voices = json.load(open("voices.json"))
lang, quality = CONFIG["LANGUAGE"], CONFIG["VOICE_QUALITY"]
VOICES = {}
for key, info in all_voices.items():
    if key.startswith(lang) and info.get("quality") == quality:
        speaker = key.replace(f"{lang}-", "").rsplit("-", 1)[0]
        VOICES[key] = (f"https://huggingface.co/rhasspy/piper-voices/resolve/main/"
                       f"{lang.split('_')[0]}/{lang}/{speaker}/{quality}/{key}.onnx")
os.makedirs("voices", exist_ok=True)
for name, url in VOICES.items():
    if not os.path.exists(f"voices/{name}.onnx"):
        run(["wget", "-q", "-O", f"voices/{name}.onnx", url], check=False)
        run(["wget", "-q", "-O", f"voices/{name}.onnx.json", url + ".json"], check=False)
VOICES = [n for n in VOICES if os.path.getsize(f"voices/{n}.onnx") > 1e6]
CONFIG["VOICES"] = VOICES

# --- The assistant's own voice (any language) ---
her = CONFIG["HER_VOICE"]
if her and not (os.path.exists(f"voices/{her}.onnx") and os.path.getsize(f"voices/{her}.onnx") > 1e6):
    if her in all_voices:
        h_lang, h_speaker, h_quality = all_voices[her]["language"]["code"], her.split("-")[1], all_voices[her]["quality"]
        url = (f"https://huggingface.co/rhasspy/piper-voices/resolve/main/"
               f"{h_lang.split('_')[0]}/{h_lang}/{h_speaker}/{h_quality}/{her}.onnx")
        run(["wget", "-q", "-O", f"voices/{her}.onnx", url], check=False)
        run(["wget", "-q", "-O", f"voices/{her}.onnx.json", url + ".json"], check=False)
    if not (os.path.exists(f"voices/{her}.onnx") and os.path.getsize(f"voices/{her}.onnx") > 1e6):
        print(f"⚠️ Couldn't get HER_VOICE {her}; her phrases will be skipped")
        her = ""
CONFIG["HER_VOICE_READY"] = her

# --- LibriTTS-R generator (many blended speakers) ---
GEN = "piper-sample-generator/models/en_US-libritts_r-medium.pt"
if CONFIG["GENERATOR_SAMPLES"] and not os.path.exists(GEN):
    run(["wget", "-q", "-O", GEN, "https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/"
                                  "en_US-libritts_r-medium.pt"], check=False)
CONFIG["GENERATOR_READY"] = bool(CONFIG["GENERATOR_SAMPLES"]) and os.path.exists(GEN) and os.path.getsize(GEN) > 1e6
if CONFIG["GENERATOR_SAMPLES"] and not CONFIG["GENERATOR_READY"]:
    print("⚠️ Couldn't get the LibriTTS-R generator; training uses the single-speaker voices only")

json.dump(CONFIG, open(f"{WORK}/config.json", "w"), indent=2)
total = len(CONFIG["VARIANTS"]) * len(VOICES) * CONFIG["SAMPLES_PER_COMBO"]
gen_total = len(CONFIG["VARIANTS"]) * CONFIG["GENERATOR_SAMPLES"] if CONFIG["GENERATOR_READY"] else 0
print(f"🗣️ {len(VOICES)} voices for {lang}/{quality}: {VOICES}")
print(f"   Training samples: {len(CONFIG['VARIANTS'])} variant(s) x {len(VOICES)} voices x "
      f"{CONFIG['SAMPLES_PER_COMBO']} = {total}, plus {gen_total} from the LibriTTS-R generator")
print(f"   Hard negatives: {len(CONFIG['SOUNDALIKES'])} sound-alikes x {len(VOICES)} voices"
      + (f", and {CONFIG['HER_SAMPLES']} replies in her voice ({her})" if her else ""))
if CONFIG["MY_CLIPS"]:
    for kind in ["positive", "negative"]:
        n = len(glob.glob(f"{CONFIG['MY_CLIPS']}/**/{kind}/*.wav", recursive=True))
        print(f"   Your recordings, {kind}: {n}" + ("" if n else "  ⚠️ none found"))

# --- Play one clip of each test variant ---
from IPython.display import Audio, display, HTML
for i, word in enumerate(CONFIG["TEST_VARIANTS"]):
    out_dir = f"{WORK}/test_{word}"
    shutil.rmtree(out_dir, ignore_errors=True)
    run([VPY, "-m", "piper_sample_generator", word, "--model", f"{WORK}/voices/{VOICES[0]}.onnx",
         "--max-samples", "1", "--output-dir", out_dir], cwd="piper-sample-generator", check=False,
        noise=("UserWarning", "warnings.warn", "FutureWarning"))
    wavs = sorted(glob.glob(f"{out_dir}/*.wav"))
    display(HTML(f"<h3>#{i}: {word}{'  (used for training)' if word in CONFIG['VARIANTS'] else ''}</h3>"))
    if wavs:
        display(Audio(wavs[0]))
    else:
        print("   ❌ No clip generated for this variant (see output above)")

Wake word: My Jeannie (mi_jeenee), language en_US
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB
💾 Free disk in /tmp/mww: 1111 GB
✅ Python 3.11 environment already set up
✅ webrtcvad OK
✅ Stride patch already applied
🐍 Python 3.11.17 | TensorFlow 2.20.0 | GPU: /physical_device:GPU:0
🗣️ 18 voices for en_US/medium: ['en_US-amy-medium', 'en_US-arctic-medium', 'en_US-bryce-medium', 'en_US-hfc_female-medium', 'en_US-hfc_male-medium', 'en_US-joe-medium', 'en_US-john-medium', 'en_US-kristin-medium', 'en_US-kusal-medium', 'en_US-l2arctic-medium', 'en_US-lessac-medium', 'en_US-libritts_r-medium', 'en_US-ljspeech-medium', 'en_US-mike-medium', 'en_US-norman-medium', 'en_US-reza_ibrahim-medium', 'en_US-ryan-medium', 'en_US-sam-medium']
   Training samples: 1 variant(s) x 18 voices x 250 = 4500
DEBUG:__main__:Loading ['/tmp/mww/voices/en_US-amy-medium.onnx']
DEBUG:piper.voice:Guessing voice config path: /tmp/mww/voices/en_US-amy-medium.onnx.json
DEBUG:piper.voice:Using CUDA
INFO:__main__:Successfully load

DEBUG:__main__:Loading ['/tmp/mww/voices/en_US-amy-medium.onnx']
DEBUG:piper.voice:Guessing voice config path: /tmp/mww/voices/en_US-amy-medium.onnx.json
DEBUG:piper.voice:Using CUDA
INFO:__main__:Successfully loaded model(s)


DEBUG:__main__:Loading ['/tmp/mww/voices/en_US-amy-medium.onnx']
DEBUG:piper.voice:Guessing voice config path: /tmp/mww/voices/en_US-amy-medium.onnx.json
DEBUG:piper.voice:Using CUDA
INFO:__main__:Successfully loaded model(s)


## Step 2: Download data

In [11]:
# STEP 2: Download augmentation data and negative datasets (run once; finished items are skipped)

import os, sys, re, glob, json, shutil, subprocess
WORK = "/kaggle/temp/mww" if os.path.isdir("/kaggle/temp") else "/tmp/mww"
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
VPY = f"{WORK}/py311/bin/python"          # the private Python 3.11
UV = [sys.executable, "-m", "uv"]

def run(cmd, check=True, cwd=None, noise=()):
    # Run a command, stream its output live, optionally hide known-harmless log lines
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, "PYTHONUNBUFFERED": "1", "TF_CPP_MIN_LOG_LEVEL": "2",
                              "MPLBACKEND": "Agg"})  # Kaggle's inline plot setting breaks matplotlib in Python 3.11
    for line in p.stdout:
        if not any(n in line for n in noise):
            print(line, end="")
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f"❌ Command failed (exit {p.returncode}): {' '.join(map(str, cmd))[:150]}")
    return p.returncode

def count_wavs(d):
    return len(glob.glob(f"{d}/*.wav")) if os.path.isdir(d) else 0

if not os.path.exists(VPY):
    raise RuntimeError("❌ Run Step 1 first.")
CONFIG = json.load(open(f"{WORK}/config.json"))
print(f"💾 Free disk: {shutil.disk_usage(WORK).free/1e9:.0f} GB")

# --- Room impulse responses, background noise and music, converted inside Python 3.11 ---
open("prep_aug.py", "w").write('''
import os, io, glob, subprocess, numpy as np, scipy.io.wavfile, soundfile as sf
from pathlib import Path
from tqdm import tqdm

def count_wavs(d):
    return len(glob.glob(f"{d}/*.wav")) if os.path.isdir(d) else 0

def write16k(path, data, sr):
    if data.ndim > 1:
        data = data[:, 0]
    if sr != 16000:
        import librosa
        data = librosa.resample(data.astype(np.float32), orig_sr=sr, target_sr=16000)
    scipy.io.wavfile.write(path, 16000, (np.clip(data, -1, 1) * 32767).astype(np.int16))

# MIT room impulse responses
if count_wavs("mit_rirs") == 0:
    print("⬇️ Room impulse responses...")
    try:
        from datasets import load_dataset, Audio
        os.makedirs("mit_rirs", exist_ok=True)
        ds = load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train")
        ds = ds.cast_column("audio", Audio(decode=False))
        for i, row in enumerate(tqdm(ds, mininterval=10)):
            a = row["audio"]
            src = io.BytesIO(a["bytes"]) if a.get("bytes") else a.get("path")
            if src is None:
                continue
            data, sr = sf.read(src)
            write16k(f"mit_rirs/rir_{i}.wav", data, sr)
    except Exception as e:
        print(f"❌ Impulse responses failed: {e}")

# AudioSet (optional; the link has failed on Kaggle before)
if count_wavs("audioset_16k") == 0:
    print("⬇️ AudioSet background noise (optional)...")
    try:
        os.makedirs("audioset", exist_ok=True)
        os.makedirs("audioset_16k", exist_ok=True)
        subprocess.run(["wget", "-q", "-O", "audioset/bal_train09.tar",
                        "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar"], check=True)
        subprocess.run(["tar", "-xf", "audioset/bal_train09.tar", "-C", "audioset"], check=True)
        for flac in tqdm(list(Path("audioset").glob("**/*.flac")), mininterval=10):
            try:
                data, sr = sf.read(str(flac))
                write16k(f"audioset_16k/{flac.stem}.wav", data, sr)
            except Exception:
                pass
    except Exception as e:
        print(f"⚠️ AudioSet failed (optional, training works without it): {e}")
    subprocess.run(["rm", "-rf", "audioset"])

# FMA music
if count_wavs("fma_16k") == 0:
    print("⬇️ FMA background music...")
    try:
        os.makedirs("fma", exist_ok=True)
        os.makedirs("fma_16k", exist_ok=True)
        if not list(Path("fma").glob("**/*.mp3")):
            subprocess.run(["wget", "-q", "-O", "fma/fma_xs.zip",
                            "https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/fma_xs.zip"], check=True)
            import zipfile
            zipfile.ZipFile("fma/fma_xs.zip").extractall("fma")
        for mp3 in tqdm(list(Path("fma").glob("**/*.mp3")), mininterval=10):
            try:
                try:
                    data, sr = sf.read(str(mp3))
                    write16k(f"fma_16k/{mp3.stem}.wav", data, sr)
                except Exception:
                    from pydub import AudioSegment
                    seg = AudioSegment.from_mp3(str(mp3)).set_frame_rate(16000).set_channels(1)
                    samples = np.array(seg.get_array_of_samples(), dtype=np.int16)
                    scipy.io.wavfile.write(f"fma_16k/{mp3.stem}.wav", 16000, samples)
            except Exception:
                pass
    except Exception as e:
        print(f"❌ FMA failed: {e}")
''')
run([VPY, "prep_aug.py"], check=False, noise=("UserWarning", "warnings.warn", "FutureWarning"))

# --- Pre-generated negative datasets ---
import zipfile
neg_root = "https://huggingface.co/datasets/kahrendt/microwakeword/resolve/main/"
needed = ["speech", "dinner_party", "no_speech"] + (["dinner_party_eval"] if CONFIG["USE_DINNER_PARTY_EVAL"] else [])
os.makedirs("negative_datasets", exist_ok=True)
for name in needed:
    if os.path.isdir(f"negative_datasets/{name}") and os.listdir(f"negative_datasets/{name}"):
        continue
    z = f"negative_datasets/{name}.zip"
    print(f"⬇️ Negative dataset: {name}...")
    run(["wget", "-q", "--progress=dot:giga", "-O", z, neg_root + f"{name}.zip"], check=False)
    try:
        zipfile.ZipFile(z).extractall("negative_datasets")
        os.remove(z)
    except Exception as e:
        print(f"❌ Couldn't unpack {name}: {e}")

# --- Summary ---
print("\n📦 Data check:")
for d, required in [("mit_rirs", True), ("fma_16k", False), ("audioset_16k", False)]:
    n = count_wavs(d)
    print(f"  {'✅' if n else ('❌' if required else '⚠️')} {d} ({n} wav files)")
print("  (Need at least one of fma_16k / audioset_16k.)")
for name in needed:
    ok = os.path.isdir(f"negative_datasets/{name}") and bool(os.listdir(f"negative_datasets/{name}"))
    print(f"  {'✅' if ok else '❌'} negative_datasets/{name}")
print(f"💾 Free disk now: {shutil.disk_usage(WORK).free/1e9:.0f} GB")
print("\n👉 If the data check looks good, run Step 3.")


💾 Free disk: 1111 GB
⬇️ AudioSet background noise (optional)...
⚠️ AudioSet failed (optional, training works without it): Command '['wget', '-q', '-O', 'audioset/bal_train09.tar', 'https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/bal_train09.tar']' returned non-zero exit status 8.

📦 Data check:
  ✅ mit_rirs (270 wav files)
  ✅ fma_16k (210 wav files)
  ⚠️ audioset_16k (0 wav files)
  (Need at least one of fma_16k / audioset_16k.)
  ✅ negative_datasets/speech
  ✅ negative_datasets/dinner_party
  ✅ negative_datasets/no_speech
💾 Free disk now: 1111 GB

👉 If the data check looks good, run Step 3.


## Step 3: Generate, train, export

In [12]:
# STEP 3: Generate samples, build features, train, and export .tflite + .json for the Voice PE
# Uses the settings from Step 1 (re-run Step 1 after changing them).

import os, sys, re, glob, json, shutil, subprocess
WORK = "/kaggle/temp/mww" if os.path.isdir("/kaggle/temp") else "/tmp/mww"
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
VPY = f"{WORK}/py311/bin/python"          # the private Python 3.11
UV = [sys.executable, "-m", "uv"]

def run(cmd, check=True, cwd=None, noise=()):
    # Run a command, stream its output live, optionally hide known-harmless log lines
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         env={**os.environ, "PYTHONUNBUFFERED": "1", "TF_CPP_MIN_LOG_LEVEL": "2",
                              "MPLBACKEND": "Agg"})  # Kaggle's inline plot setting breaks matplotlib in Python 3.11
    for line in p.stdout:
        if not any(n in line for n in noise):
            print(line, end="")
    p.wait()
    if check and p.returncode != 0:
        raise RuntimeError(f"❌ Command failed (exit {p.returncode}): {' '.join(map(str, cmd))[:150]}")
    return p.returncode

def count_wavs(d):
    return len(glob.glob(f"{d}/*.wav")) if os.path.isdir(d) else 0

if not os.path.exists(VPY):
    raise RuntimeError("❌ Run Step 1 and Step 2 first.")
CONFIG = json.load(open(f"{WORK}/config.json"))
NAME = CONFIG["WAKE_WORD_NAME"]
print(f"📓 Notebook {CONFIG.get('NOTEBOOK', 'older than 2026-10-23c')}; your recordings: {CONFIG.get('MY_CLIPS') or 'none'}")
if CONFIG.get("NOTEBOOK") != "2026-10-23c":
    raise RuntimeError("❌ The settings Step 3 found were saved by another notebook's Step 1: run Step 1 again (then Step 2).")

# --- Preflight ---
problems = []
if count_wavs("mit_rirs") == 0:
    problems.append("mit_rirs has no wav files")
background = [d for d in ["fma_16k", "audioset_16k"] if count_wavs(d) > 0]
if not background:
    problems.append("no background audio (fma_16k / audioset_16k)")
for name in ["speech", "dinner_party", "no_speech"] + (["dinner_party_eval"] if CONFIG["USE_DINNER_PARTY_EVAL"] else []):
    if not (os.path.isdir(f"negative_datasets/{name}") and os.listdir(f"negative_datasets/{name}")):
        problems.append(f"negative_datasets/{name} missing")
if not CONFIG.get("VOICES"):
    problems.append("no voices downloaded (re-run Step 1)")
if problems:
    raise RuntimeError("❌ Not ready. Re-run Step 2 and check its data check:\n  - " + "\n  - ".join(problems))
print(f"✅ Preflight passed. Background audio: {background}")

# --- Your own recordings: copied in so they can be counted and stamped ---
import wave
import numpy as np

def find_word(src, dst):
    # A positive clip cut to CLIP_DURATION_MS ending a quarter second after its loudest MY_CLIPS_WORD_SECONDS (the
    # word); copied as is if it isn't 16 kHz mono 16-bit, or is already that short
    with wave.open(src) as w:
        rate, ch, width, raw = w.getframerate(), w.getnchannels(), w.getsampwidth(), w.readframes(w.getnframes())
    keep = int(rate * CONFIG["CLIP_DURATION_MS"] / 1000)
    x = np.frombuffer(raw, dtype=np.int16).astype(np.float64) if width == 2 else None
    if rate != 16000 or ch != 1 or x is None or len(x) <= keep:
        if rate != 16000 or ch != 1 or width != 2:
            print(f"⚠️ {os.path.basename(src)} isn't 16 kHz mono 16-bit: copied as is")
        shutil.copy(src, dst)
        return
    step = rate // 20                                            # 50 ms
    energy = np.sqrt(np.mean(x[: len(x) // step * step].reshape(-1, step) ** 2, axis=1))
    span = max(1, int(CONFIG.get("MY_CLIPS_WORD_SECONDS", 0.8) * 20))
    loud = np.convolve(energy, np.ones(span), mode="valid")       # the loudness of each WORD_SECONDS stretch
    end = min(len(x), (int(np.argmax(loud)) + span) * step + rate // 4)
    start = max(0, end - keep)
    end = start + keep
    with wave.open(dst, "wb") as o:
        o.setnchannels(1); o.setsampwidth(2); o.setframerate(rate)
        o.writeframes(raw[start * 2: end * 2])

my_counts = {}
for kind in ["positive", "negative"]:
    files = sorted(glob.glob(f"{CONFIG['MY_CLIPS']}/**/{kind}/*.wav", recursive=True)) if CONFIG["MY_CLIPS"] else []
    my_counts[kind] = len(files)
    dest = f"my_clips/{kind}"
    if True:                                     # always afresh: cheap, and MY_CLIPS_FIND_WORD may have changed
        shutil.rmtree(dest, ignore_errors=True)
        os.makedirs(dest, exist_ok=True)
        for i, f in enumerate(files):
            if kind == "positive" and CONFIG.get("MY_CLIPS_FIND_WORD", True):
                find_word(f, f"{dest}/{i}.wav")
            else:
                shutil.copy(f, f"{dest}/{i}.wav")
if CONFIG["MY_CLIPS"]:
    print(f"🎙️ Your recordings: {my_counts['positive']} positive, {my_counts['negative']} negative")

# --- Start fresh if the sample/training settings changed since the last run ---
stamp_keys = ["WAKE_WORD_NAME", "VARIANTS", "SAMPLES_PER_COMBO", "GENERATOR_SAMPLES", "GENERATOR_READY", "LANGUAGE",
              "VOICE_QUALITY", "CLIP_DURATION_MS", "SOUNDALIKES", "SOUNDALIKE_SAMPLES_PER_VOICE", "HER_VOICE_READY",
              "HER_PHRASES", "HER_SAMPLES", "MY_CLIPS_REPEAT", "MY_CLIPS_FIND_WORD", "NEGATIVE_CLASS_WEIGHT", "TRAINING_STEPS",
              "FINE_TUNE_STEPS", "SPEC_AUGMENT", "USE_DINNER_PARTY_EVAL"]
stamp = {k: CONFIG.get(k) for k in stamp_keys}
stamp["MY_CLIPS_COUNTS"] = my_counts
try:
    old_stamp = json.load(open("run_stamp.json"))
except Exception:
    old_stamp = None
expected = len(CONFIG["VARIANTS"]) * len(CONFIG["VOICES"]) * CONFIG["SAMPLES_PER_COMBO"]
if CONFIG["GENERATOR_READY"]:
    expected += len(CONFIG["VARIANTS"]) * CONFIG["GENERATOR_SAMPLES"]
FEATURE_DIRS = ["generated_augmented_features", "hard_negative_features", "my_positive_features",
                "my_negative_features"]
if old_stamp != stamp:
    for d in ["generated_samples", "hard_negatives", f"trained_models/{NAME}"] + FEATURE_DIRS:
        shutil.rmtree(d, ignore_errors=True)
    if old_stamp is not None:
        print("🧹 Settings changed: clearing old samples, features and checkpoints")

NOISE = ("UserWarning", "warnings.warn", "FutureWarning", "cuda_dnn", "cuda_blas", "cuda_fft",
         "computation_placer", "absl::InitializeLog", "WARNING: All log")

def generate(text, model, count, out_dir, start, extra=()):
    # Piper writes 0.wav, 1.wav, ... into a temp folder; they are renamed into out_dir from `start` on
    tmp = f"{WORK}/tmp_gen"
    shutil.rmtree(tmp, ignore_errors=True)
    run([VPY, "-m", "piper_sample_generator", text, "--model", model, "--max-samples", str(count),
         "--output-dir", tmp, *extra], cwd="piper-sample-generator", check=False, noise=NOISE)
    for f in sorted(glob.glob(f"{tmp}/*.wav")):
        os.rename(f, f"{out_dir}/{start}.wav")
        start += 1
    return start

# --- 1) Generate wake word samples ---
os.makedirs("generated_samples", exist_ok=True)
if count_wavs("generated_samples") < expected * 0.95:
    shutil.rmtree("generated_samples"); os.makedirs("generated_samples")
    print(f"\n🚀 Generating {expected} samples...")
    idx = 0
    for word in CONFIG["VARIANTS"]:
        for voice in CONFIG["VOICES"]:
            print(f"   {word} / {voice}")
            idx = generate(word, f"{WORK}/voices/{voice}.onnx", CONFIG["SAMPLES_PER_COMBO"], f"{WORK}/generated_samples",
                           idx, ["--batch-size", "50"])
        if CONFIG["GENERATOR_READY"]:
            print(f"   {word} / LibriTTS-R generator")
            idx = generate(word, "models/en_US-libritts_r-medium.pt", CONFIG["GENERATOR_SAMPLES"],
                           f"{WORK}/generated_samples", idx, ["--batch-size", "100"])
    print(f"✅ {idx} samples generated")
else:
    print(f"✅ Samples already generated ({count_wavs('generated_samples')})")
if count_wavs("generated_samples") == 0:
    raise RuntimeError("❌ No samples were generated (see generator output above).")

# --- 1b) Generate hard negatives: sound-alikes in every voice, and replies in her own voice ---
if count_wavs("hard_negatives") == 0:
    os.makedirs("hard_negatives", exist_ok=True)
    print("\n🚀 Generating hard negatives...")
    idx = 0
    if CONFIG["SOUNDALIKES"]:
        open(f"{WORK}/soundalikes.txt", "w").write("\n".join(CONFIG["SOUNDALIKES"]) + "\n")
        for voice in CONFIG["VOICES"]:
            idx = generate(f"{WORK}/soundalikes.txt", f"{WORK}/voices/{voice}.onnx",
                           CONFIG["SOUNDALIKE_SAMPLES_PER_VOICE"], f"{WORK}/hard_negatives", idx)
    if CONFIG["HER_VOICE_READY"] and CONFIG["HER_PHRASES"]:
        print(f"   her voice: {CONFIG['HER_VOICE_READY']}")
        open(f"{WORK}/her_phrases.txt", "w").write("\n".join(CONFIG["HER_PHRASES"]) + "\n")
        idx = generate(f"{WORK}/her_phrases.txt", f"{WORK}/voices/{CONFIG['HER_VOICE_READY']}.onnx",
                       CONFIG["HER_SAMPLES"], f"{WORK}/hard_negatives", idx)
    print(f"✅ {idx} hard negatives generated")
else:
    print(f"✅ Hard negatives already generated ({count_wavs('hard_negatives')})")

# --- 2) Spectrogram features ---
# Each job: (input folder, output folder, augmenter, use the train/validation/test split, training repeats)
jobs = [("generated_samples", "generated_augmented_features", "synthetic", True, 2)]
if count_wavs("hard_negatives"):
    jobs.append(("hard_negatives", "hard_negative_features", "synthetic", True, 1))
if count_wavs("my_clips/positive"):
    jobs.append(("my_clips/positive", "my_positive_features", "real", False, CONFIG["MY_CLIPS_REPEAT"]))
if count_wavs("my_clips/negative"):
    jobs.append(("my_clips/negative", "my_negative_features", "real", False, CONFIG["MY_CLIPS_REPEAT"]))
jobs = [j for j in jobs if not os.path.isdir(f"{j[1]}/training/wakeword_mmap")]
if jobs:
    print(f"\n🚀 Building spectrogram features: {', '.join(j[1] for j in jobs)}...")
    for j in jobs:
        shutil.rmtree(j[1], ignore_errors=True)
    open("prep_features.py", "w").write(f'''
import os
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
from mmap_ninja.ragged import RaggedMmap

augmenters = {{
    # Piper samples: clean, so add plenty: room echo, loud background, distortion (shouting), volume swings
    "synthetic": Augmentation(
        augmentation_duration_s=3.2,
        augmentation_probabilities={{"SevenBandParametricEQ": 0.1, "TanhDistortion": 0.2, "PitchShift": 0.1,
                                     "BandStopFilter": 0.1, "AddColorNoise": 0.1, "AddBackgroundNoise": 0.75,
                                     "Gain": 1.0, "GainTransition": 0.25, "RIR": 0.5}},
        impulse_paths=["mit_rirs"], background_paths={background!r},
        background_min_snr_db=-5, background_max_snr_db=10, min_jitter_s=0.195, max_jitter_s=0.205),
    # Real recordings already have the room, the mic and its noise: vary them lightly, no echo, no added silence
    "real": Augmentation(
        augmentation_duration_s=3.2,
        augmentation_probabilities={{"SevenBandParametricEQ": 0.1, "TanhDistortion": 0.1, "PitchShift": 0.1,
                                     "BandStopFilter": 0.1, "AddColorNoise": 0.1, "AddBackgroundNoise": 0.3,
                                     "Gain": 1.0, "GainTransition": 0.25, "RIR": 0.0}},
        impulse_paths=["mit_rirs"], background_paths={background!r},
        background_min_snr_db=5, background_max_snr_db=15, min_gain_db=-15, max_gain_db=0),
}}

for input_dir, out_root, kind, split_it, repeat in {jobs!r}:
    clips = Clips(input_directory=input_dir, file_pattern="*.wav", max_clip_duration_s=None, remove_silence=False,
                  random_split_seed=10 if split_it else None, split_count=0.1)
    splits = ["training", "validation", "testing"] if split_it else ["training"]
    for split in splits:
        out_dir = os.path.join(out_root, split)
        os.makedirs(out_dir, exist_ok=True)
        split_name = {{"training": "train", "validation": "validation", "testing": "test"}}[split] if split_it else None
        spectrograms = SpectrogramGeneration(clips=clips, augmenter=augmenters[kind],
                                             slide_frames=1 if split == "testing" else 10, step_ms=10)
        RaggedMmap.from_generator(out_dir=os.path.join(out_dir, "wakeword_mmap"),
                                  sample_generator=spectrograms.spectrogram_generator(
                                      split=split_name, repeat=repeat if split == "training" else 1),
                                  batch_size=100, verbose=True)
    print(f"{{out_root}} ready")
''')
    run([VPY, "prep_features.py"], noise=NOISE)
else:
    print("✅ Features already built")

# --- 3) Training config (JSON is valid YAML) ---
def feature_set(path, truth, weight, truncation):
    return {"features_dir": path, "sampling_weight": weight, "penalty_weight": 1.0, "truth": truth,
            "truncation_strategy": truncation, "type": "mmap"}

features = [feature_set("generated_augmented_features", True, 2.0, "truncate_start")]
if os.path.isdir("my_positive_features"):
    features.append(feature_set("my_positive_features", True, 0.5, "truncate_start"))
if os.path.isdir("hard_negative_features"):
    # truncate_start like the positives: the end of a sound-alike lines up where the wake word would end
    features.append(feature_set("hard_negative_features", False, 3.0, "truncate_start"))
if os.path.isdir("my_negative_features"):
    features.append(feature_set("my_negative_features", False, 1.0, "random"))
features += [
    feature_set("negative_datasets/speech", False, 10.0, "random"),
    feature_set("negative_datasets/dinner_party", False, 10.0, "random"),
    feature_set("negative_datasets/no_speech", False, 5.0, "random"),
]
if CONFIG["USE_DINNER_PARTY_EVAL"]:
    features.append(feature_set("negative_datasets/dinner_party_eval", False, 1.0, "split"))

# One or two phases: the second fine-tunes at a tenth of the learning rate
steps = [CONFIG["TRAINING_STEPS"]] + ([CONFIG["FINE_TUNE_STEPS"]] if CONFIG["FINE_TUNE_STEPS"] else [])
per_phase = lambda v: [v] * len(steps)
mask_size, mask_count = (5, 2) if CONFIG["SPEC_AUGMENT"] else (0, 0)
train_cfg = {
    "window_step_ms": 10, "train_dir": f"trained_models/{NAME}", "features": features,
    "training_steps": steps, "positive_class_weight": per_phase(1),
    "negative_class_weight": per_phase(CONFIG["NEGATIVE_CLASS_WEIGHT"]), "learning_rates": [0.001, 0.0001][:len(steps)],
    "batch_size": 128,
    "time_mask_max_size": per_phase(mask_size), "time_mask_count": per_phase(mask_count),
    "freq_mask_max_size": per_phase(mask_size), "freq_mask_count": per_phase(mask_count),
    "eval_step_interval": CONFIG["EVAL_STEP_INTERVAL"], "clip_duration_ms": CONFIG["CLIP_DURATION_MS"],
    "target_minimization": 0.9, "minimization_metric": None, "maximization_metric": "average_viable_recall",
}
json.dump(train_cfg, open("training_parameters.yaml", "w"), indent=2)
json.dump(stamp, open("run_stamp.json", "w"))
print("Feature sets: " + ", ".join(f"{f['features_dir']} ({'wake' if f['truth'] else 'not'})" for f in features))

# --- 4) Train ---
print(f"\n🚀 Training {' + '.join(map(str, steps))} steps (roughly 1-2 hours on a T4; may look quiet for a while)...")
run([VPY, "-m", "microwakeword.model_train_eval", "--training_config=training_parameters.yaml",
     "--train", "1", "--restore_checkpoint", "1",
     "--test_tf_nonstreaming", "0", "--test_tflite_nonstreaming", "0", "--test_tflite_nonstreaming_quantized", "0",
     "--test_tflite_streaming", "0", "--test_tflite_streaming_quantized", "1", "--use_weights", "best_weights",
     "mixednet", "--pointwise_filters", "64,64,64,64", "--repeat_in_block", "1, 1, 1, 1",
     "--mixconv_kernel_sizes", "[5], [7,11], [9,15], [23]", "--residual_connection", "0,0,0,0",
     "--first_conv_filters", "32", "--first_conv_kernel_size", "5", "--stride", "3"], noise=NOISE)

# --- 5) The cutoff: misses (frr) and false wakes per hour (faph) at each cutoff, from the dinner party test ---
roc = f"trained_models/{NAME}/tflite_stream_state_internal_quant/tflite_streaming_roc.txt"
rows = []
if os.path.exists(roc):
    shutil.copy(roc, f"/kaggle/working/{NAME}_cutoffs.txt")
    for line in open(roc):
        m = re.match(r"Cutoff ([\d.]+): frr=([\d.]+); faph=([\d.]+)", line.strip())
        if m:
            rows.append(tuple(float(x) for x in m.groups()))
cutoff = CONFIG["PROBABILITY_CUTOFF"]
if cutoff == "auto":
    # Never a cutoff that misses most wake words: at 1.00 the model hears nothing, so it has no false wakes either
    max_misses = CONFIG.get("MAX_MISSES", 0.10)
    hears = [r for r in rows if r[1] <= max_misses]
    viable = [r for r in hears if r[2] <= CONFIG["TARGET_FALSE_WAKES_PER_HOUR"]]
    if not (CONFIG["USE_DINNER_PARTY_EVAL"] and rows):
        cutoff = CONFIG["FALLBACK_CUTOFF"]
        print(f"\n⚠️ No false-wake table (USE_DINNER_PARTY_EVAL off?); using FALLBACK_CUTOFF {cutoff}")
    elif viable:
        best = min(viable, key=lambda r: r[0])
        cutoff = round(best[0], 2)
        print(f"\n🎯 Cutoff {cutoff}: misses {best[1]*100:.1f}% of wake words, "
              f"{best[2]:.2f} false wakes per hour in the dinner party test")
    elif hears:
        best = min(hears, key=lambda r: (r[2], r[1]))
        cutoff = round(best[0], 2)
        print(f"\n⚠️ No cutoff reaches {CONFIG['TARGET_FALSE_WAKES_PER_HOUR']} false wakes per hour while missing at most "
              f"{max_misses*100:.0f}% of wake words. Using {cutoff}, the fewest false wakes that still hears it: misses "
              f"{best[1]*100:.1f}%, {best[2]:.2f} false wakes per hour in the dinner party test. More negatives (your own "
              f"recordings in MY_CLIPS, more SOUNDALIKES) or a higher NEGATIVE_CLASS_WEIGHT should bring it down.")
    else:
        cutoff = CONFIG["FALLBACK_CUTOFF"]
        print(f"\n⚠️ Every cutoff misses more than {max_misses*100:.0f}% of wake words: the model didn't learn the "
              f"word. Using FALLBACK_CUTOFF {cutoff}; check the samples (Step 1's test clips) and train again.")
if rows:
    print("   cutoff   misses   false wakes/hour")
    for c, frr, faph in rows:
        print(f"   {c:.2f}    {frr*100:5.1f}%    {faph:.2f}")

# --- 6) Export .tflite + .json manifest ---
src = f"trained_models/{NAME}/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite"
if not os.path.exists(src):
    found = glob.glob(f"trained_models/{NAME}/**/*.tflite", recursive=True)
    if not found:
        raise RuntimeError("❌ Training finished but no .tflite was found.")
    src = found[0]
out_tflite, out_json = f"/kaggle/working/{NAME}.tflite", f"/kaggle/working/{NAME}.json"
shutil.copy(src, out_tflite)
manifest = {
    "type": "micro",
    "wake_word": CONFIG["WAKE_WORD_DISPLAY"],
    "author": "custom",
    "website": "https://github.com/kahrendt/microWakeWord",
    "model": f"{NAME}.tflite",
    "trained_languages": [CONFIG["LANGUAGE"].split("_")[0]],
    "version": 2,
    "micro": {
        "probability_cutoff": cutoff,
        "feature_step_size": 10,
        "sliding_window_size": 5,
        "tensor_arena_size": 30000,
        "minimum_esphome_version": "2024.7.0",
    },
}
json.dump(manifest, open(out_json, "w"), indent=2)

from IPython.display import FileLink, display
print(f"\n✅ Done! {NAME}.tflite ({os.path.getsize(out_tflite)/1e3:.0f} KB) and {NAME}.json are in /kaggle/working")
display(FileLink(f"{NAME}.tflite")); display(FileLink(f"{NAME}.json"))
if rows:
    display(FileLink(f"{NAME}_cutoffs.txt"))
print("⬇️ Download them from the Output tab. Keep the .tflite and .json together: the .json points to the .tflite by name.")

✅ Preflight passed. Background audio: ['fma_16k']

🚀 Generating 4500 samples...
   mi_jeenee / en_US-amy-medium
DEBUG:__main__:Loading ['/tmp/mww/voices/en_US-amy-medium.onnx']
DEBUG:piper.voice:Guessing voice config path: /tmp/mww/voices/en_US-amy-medium.onnx.json
DEBUG:piper.voice:Using CUDA
INFO:__main__:Successfully loaded model(s)
   mi_jeenee / en_US-arctic-medium
DEBUG:__main__:Loading ['/tmp/mww/voices/en_US-arctic-medium.onnx']
DEBUG:piper.voice:Guessing voice config path: /tmp/mww/voices/en_US-arctic-medium.onnx.json
DEBUG:piper.voice:Using CUDA
INFO:__main__:Successfully loaded model(s)
   mi_jeenee / en_US-bryce-medium
DEBUG:__main__:Loading ['/tmp/mww/voices/en_US-bryce-medium.onnx']
DEBUG:piper.voice:Guessing voice config path: /tmp/mww/voices/en_US-bryce-medium.onnx.json
DEBUG:piper.voice:Using CUDA
INFO:__main__:Successfully loaded model(s)
   mi_jeenee / en_US-hfc_female-medium
DEBUG:__main__:Loading ['/tmp/mww/voices/en_US-hfc_female-medium.onnx']
DEBUG:piper.voice:Gu

/tmp/mww/mi_jeenee.tflite

/tmp/mww/mi_jeenee.json

⬇️ Download both from the Output tab. Keep them together: the .json points to the .tflite by name.
